In [1]:
# ==========================================
# Notebook 13
# Recommendation Evaluation Engine
# ==========================================

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
profiles_df = pd.read_csv("../data/item_profiles.csv")

profiles_df.head()

,item_id,title,category,tags,content_profile
0,1,The Early Days of Stripe,Startup,startup fintech founders entrepreneurship vent...,Title: The Early Days of Stripe Category: Star...
1,2,Building SpaceX,Startup,startup fintech founders entrepreneurship vent...,Title: Building SpaceX Category: Startup Tags:...
2,3,AI for Healthcare,Artificial Intelligence,ai machine-learning llm deep-learning transfor...,Title: AI for Healthcare Category: Artificial ...
3,4,The Psychology of Habits,Wellness,habits productivity self-improvement,Title: The Psychology of Habits Category: Well...
4,5,Cloud Computing Fundamentals,Technology,cloud computing distributed-systems scalability,Title: Cloud Computing Fundamentals Category: ...


In [6]:
interactions_df = pd.read_csv("../data/user_interactions.csv")

interactions_df.head()

,user_id,item_id,rating
0,101,1,5
1,101,2,5
2,101,5,4
3,102,1,5
4,102,8,5


In [3]:
hybrid_df = pd.read_csv("../data/hybrid_recommendations.csv")

hybrid_df.head()

,item_id,title,hybrid_score,user_id
0,8,Startup Fundraising Guide,1.556002,101
1,6,Modern Data Engineering,0.590070,101
2,7,Machine Learning in Finance,0.519143,101
3,9,Nutrition Science,0.181014,101
4,4,The Psychology of Habits,0.124730,101


In [4]:
item_embeddings = np.load("../data/item_embeddings.npy")

item_embeddings.shape

(10, 384)

In [7]:
print("Users:", interactions_df["user_id"].nunique())

print()

print("Items:", profiles_df["item_id"].nunique())

print()

print("Recommendations:", len(hybrid_df))

Users: 5

Items: 10

Recommendations: 25


In [8]:
ground_truth = interactions_df[interactions_df["rating"] >= 4]

ground_truth.head()

,user_id,item_id,rating
0,101,1,5
1,101,2,5
2,101,5,4
3,102,1,5
4,102,8,5


In [9]:
def precision_at_k(recommended_items, relevant_items, k=5):

    recommended_items = recommended_items[:k]

    hits = len(set(recommended_items) & set(relevant_items))

    return hits / k

In [10]:
user_id = 101

recommended_items = hybrid_df[hybrid_df["user_id"] == user_id]["item_id"].tolist()

relevant_items = ground_truth[ground_truth["user_id"] == user_id]["item_id"].tolist()

precision_at_k(recommended_items, relevant_items, k=5)

0.0

In [11]:
def recall_at_k(recommended_items, relevant_items, k=5):

    recommended_items = recommended_items[:k]

    hits = len(set(recommended_items) & set(relevant_items))

    if len(relevant_items) == 0:

        return 0

    return hits / len(relevant_items)

In [12]:
recall_at_k(recommended_items, relevant_items, k=5)

0.0

In [13]:
def dcg_at_k(relevance_scores, k=5):

    relevance_scores = np.array(relevance_scores[:k])

    return np.sum(relevance_scores / np.log2(np.arange(2, len(relevance_scores) + 2)))

In [14]:
def ndcg_at_k(recommended_items, relevant_items, k=5):

    relevance_scores = []

    for item in recommended_items[:k]:

        if item in relevant_items:

            relevance_scores.append(1)

        else:

            relevance_scores.append(0)

    dcg = dcg_at_k(relevance_scores, k)

    ideal = sorted(relevance_scores, reverse=True)

    idcg = dcg_at_k(ideal, k)

    if idcg == 0:

        return 0

    return dcg / idcg

In [15]:
ndcg_at_k(recommended_items, relevant_items, k=5)

0

In [16]:
recommended_items = hybrid_df["item_id"].unique()

In [17]:
catalog_items = profiles_df["item_id"].unique()

In [18]:
coverage = len(recommended_items) / len(catalog_items)

coverage

1.0

In [19]:
def diversity_score(item_ids):

    indices = []

    for item_id in item_ids:

        idx = profiles_df[profiles_df["item_id"] == item_id].index[0]

        indices.append(idx)

    vectors = item_embeddings[indices]

    similarity_matrix = cosine_similarity(vectors)

    avg_similarity = np.mean(similarity_matrix)

    return 1 - avg_similarity

In [20]:
user_items = hybrid_df[hybrid_df["user_id"] == 101]["item_id"].tolist()

diversity_score(user_items)

0.513414740562439

In [21]:
item_popularity = interactions_df.groupby("item_id").size()

In [22]:
def novelty_score(item_ids):

    novelty_scores = []

    for item_id in item_ids:

        popularity = item_popularity.get(item_id, 1)

        novelty_scores.append(1 / popularity)

    return np.mean(novelty_scores)

In [23]:
novelty_score(user_items)

0.9

In [24]:
evaluation_results = []

In [25]:
for user_id in hybrid_df["user_id"].unique():

    recommended_items = hybrid_df[hybrid_df["user_id"] == user_id]["item_id"].tolist()

    relevant_items = ground_truth[ground_truth["user_id"] == user_id][
        "item_id"
    ].tolist()

    evaluation_results.append(
        {
            "user_id": user_id,
            "precision_at_5": precision_at_k(recommended_items, relevant_items),
            "recall_at_5": recall_at_k(recommended_items, relevant_items),
            "ndcg_at_5": ndcg_at_k(recommended_items, relevant_items),
            "diversity": diversity_score(recommended_items),
            "novelty": novelty_score(recommended_items),
        }
    )

In [26]:
evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,user_id,precision_at_5,recall_at_5,ndcg_at_5,diversity,novelty
0,101,0.0,0.0,0,0.513415,0.9
1,102,0.0,0.0,0,0.543107,0.8
2,103,0.0,0.0,0,0.503253,0.8
3,104,0.0,0.0,0,0.557056,0.8
4,105,0.0,0.0,0,0.505819,0.8


In [27]:
evaluation_df.mean(numeric_only=True)

user_id           103.00000
precision_at_5      0.00000
recall_at_5         0.00000
ndcg_at_5           0.00000
diversity           0.52453
novelty             0.82000
dtype: float64

In [28]:
evaluation_df.sort_values(by="precision_at_5", ascending=False)

,user_id,precision_at_5,recall_at_5,ndcg_at_5,diversity,novelty
0,101,0.0,0.0,0,0.513415,0.9
1,102,0.0,0.0,0,0.543107,0.8
2,103,0.0,0.0,0,0.503253,0.8
3,104,0.0,0.0,0,0.557056,0.8
4,105,0.0,0.0,0,0.505819,0.8


In [29]:
summary_metrics = pd.DataFrame(
    {
        "metric": [
            "coverage",
            "avg_precision",
            "avg_recall",
            "avg_ndcg",
            "avg_diversity",
            "avg_novelty",
        ],
        "value": [
            coverage,
            evaluation_df["precision_at_5"].mean(),
            evaluation_df["recall_at_5"].mean(),
            evaluation_df["ndcg_at_5"].mean(),
            evaluation_df["diversity"].mean(),
            evaluation_df["novelty"].mean(),
        ],
    }
)

In [30]:
summary_metrics

,metric,value
0,coverage,1.00000
1,avg_precision,0.00000
2,avg_recall,0.00000
3,avg_ndcg,0.00000
4,avg_diversity,0.52453
5,avg_novelty,0.82000


In [31]:
evaluation_df.to_csv("../data/recommendation_evaluation.csv", index=False)

In [32]:
summary_metrics.to_csv("../data/recommendation_summary_metrics.csv", index=False)

In [33]:
saved_df = pd.read_csv("../data/recommendation_evaluation.csv")

saved_df.head()

,user_id,precision_at_5,recall_at_5,ndcg_at_5,diversity,novelty
0,101,0.0,0.0,0,0.513415,0.9
1,102,0.0,0.0,0,0.543107,0.8
2,103,0.0,0.0,0,0.503253,0.8
3,104,0.0,0.0,0,0.557056,0.8
4,105,0.0,0.0,0,0.505819,0.8
